In [1]:
from datetime import datetime
import pandas as pd

_tdy = datetime.today().strftime("%Y-%m-%d")

out = pd.read_csv(f"../results/{_tdy}/all_matchups_pts_predictions.csv")


FileNotFoundError: [Errno 2] No such file or directory: '../results/2026-02-25/all_matchups_pts_predictions.csv'

In [ ]:
out = out.sort_values('pred_pts', ascending=False).head(20)
out.to_csv("./test_pts.csv", index=False  )

out

,player,team,opp,is_home,pred_fg2a,pred_fg2_rate,pred_fta,pred_ft_rate,pred_fg3a,pred_fg3_rate,pred_pts,sd_pts,baseline_pts,delta_pts,p_over_baseline_3
13,Tyrese Maxey,PHI,ATL,1,11.601117,0.503546,5.716504,0.762699,8.779905,0.347643,25.200188,5.527721,28.398735,-3.198546,0.107576
108,Kawhi Leonard,LAC,DEN,1,11.980065,0.539431,6.219664,0.840508,6.640523,0.351944,25.163793,5.135214,27.024947,-1.861154,0.144251
43,Jalen Brunson,NYK,DET,1,11.449343,0.502393,6.449822,0.806183,7.088086,0.329909,23.719146,5.153589,26.680543,-2.961397,0.119762
80,Jaylen Brown,BOS,GSW,0,13.006610,0.513595,6.681735,0.644602,5.749826,0.349608,23.697867,5.127806,29.277930,-5.580063,0.040653
6,Donovan Mitchell,CLE,BKN,1,9.693637,0.564590,5.950416,0.824583,7.962755,0.327473,23.675228,5.116213,28.190383,-4.515156,0.059070
109,Nikola Jokić,DEN,LAC,0,10.692054,0.597802,6.277889,0.735821,4.976512,0.359077,22.763712,4.670224,26.798958,-4.035246,0.083305
0,Kevin Durant,HOU,CHA,0,10.502555,0.543581,6.241552,0.730866,5.458811,0.363399,21.930878,4.797506,24.998288,-3.067410,0.119412
44,Cade Cunningham,DET,NYK,0,11.640651,0.464430,6.686540,0.728535,5.661731,0.342047,21.493657,4.936775,25.497672,-4.004014,0.070376
14,Jalen Johnson,ATL,PHI,0,11.601117,0.537516,5.919399,0.641430,4.849309,0.333908,21.126109,4.754389,23.025039,-1.898930,0.123178
110,Jamal Murray,DEN,LAC,0,10.472783,0.465753,4.583203,0.760267,7.353162,0.347731,20.910680,5.125349,24.947162,-4.036482,0.079464


In [ ]:
pd.read_csv("../data/all_gamelogs_combined.csv").columns

C:\Users\micha\AppData\Local\Temp\ipykernel_24448\2362672535.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv("../data/all_gamelogs_combined.csv").columns


Index(['player', 'date', 'season', 'mp', 'team', 'opp', 'fg', 'fga', 'fg3',
       'fg3a', 'ft', 'fta', 'orb', 'drb', 'trb', 'ast', 'stl', 'blk', 'tov',
       'pf', 'pts', 'is_home', 'is_win', 'mp_minutes', 'usage'],
      dtype='object')

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd


# ============================================================
# POINTS TICKET SELECTORS
# expects output from predict_game_points (+ optional add_prob_pts_ge_line)
# ============================================================

def _require_cols(df: pd.DataFrame, req: set[str]) -> None:
    missing = sorted(req - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


def add_matchup_key(df: pd.DataFrame) -> pd.DataFrame:
    """Adds matchup_key as AWAY@HOME."""
    out = df.copy()
    out["matchup_key"] = np.where(
        out["is_home"].astype(int) == 1,
        out["opp"].astype(str) + "@" + out["team"].astype(str),   # away@home
        out["team"].astype(str) + "@" + out["opp"].astype(str),   # away@home
    )
    return out


# ------------------------------------------------------------
# 1) "Over Line" ticket (sportsbook line based)
# ------------------------------------------------------------
def select_over_line_ticket(
    df: pd.DataFrame,
    *,
    n_legs: int = 10,
    p_col: str = "p_over_22_5",          # column created by add_prob_pts_ge_line
    min_pred_pts: float = 16.0,
    min_prob: float = 0.58,
    max_per_team: int = 3,
    rank_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Picks n legs for a points prop line using a probability column (p_over_*).
    Filters: pred_pts >= min_pred_pts AND p_col >= min_prob
    Ranks: p_col, delta_pts, pred_pts (default)
    """
    req = {"player","team","opp","is_home","pred_pts","delta_pts", p_col}
    _require_cols(df, req)

    out = add_matchup_key(df)

    pool = out[
        (out["pred_pts"] >= float(min_pred_pts)) &
        (out[p_col] >= float(min_prob))
    ].copy()

    if pool.empty:
        return pool

    if rank_cols is None:
        rank_cols = [p_col, "delta_pts", "pred_pts"]

    pool = pool.sort_values(rank_cols, ascending=[False] * len(rank_cols))

    if max_per_team is not None:
        pool["_team_rank"] = pool.groupby("team").cumcount()
        pool = pool[pool["_team_rank"] < int(max_per_team)].copy()
        pool.drop(columns=["_team_rank"], inplace=True)

    return pool.head(int(n_legs)).reset_index(drop=True)


# ------------------------------------------------------------
# 2) "Jackpot" ticket (big delta vs baseline)
# ------------------------------------------------------------
def select_jackpot_pts_ticket(
    df: pd.DataFrame,
    *,
    n_legs: int = 3,
    over_baseline_col: str = "p_over_baseline_3",
    min_pred_pts: float = 18.0,
    min_p_over_baseline: float = 0.18,
    min_delta_pts: float = 4.0,
    max_per_team: int = 1,
) -> pd.DataFrame:
    """
    Jackpot = players projected meaningfully above baseline.
    Filters:
      pred_pts >= min_pred_pts
      delta_pts >= min_delta_pts
      over_baseline_col >= min_p_over_baseline
    Ranks: over_baseline_col, delta_pts, pred_pts
    """
    req = {"player","team","opp","is_home","pred_pts","baseline_pts","delta_pts", over_baseline_col}
    _require_cols(df, req)

    out = add_matchup_key(df)

    pool = out[
        (out["pred_pts"] >= float(min_pred_pts)) &
        (out["delta_pts"] >= float(min_delta_pts)) &
        (out[over_baseline_col] >= float(min_p_over_baseline))
    ].copy()

    if pool.empty:
        return pool

    pool = pool.sort_values(
        [over_baseline_col, "delta_pts", "pred_pts"],
        ascending=[False, False, False],
    )

    if max_per_team is not None:
        pool["_team_rank"] = pool.groupby("team").cumcount()
        pool = pool[pool["_team_rank"] < int(max_per_team)].copy()
        pool.drop(columns=["_team_rank"], inplace=True)

    return pool.head(int(n_legs)).reset_index(drop=True)


# ------------------------------------------------------------
# 3) Matchup coverage ticket (2+ per matchup + insurance)
# ------------------------------------------------------------
def select_matchup_coverage_pts_ticket(
    df: pd.DataFrame,
    *,
    players_per_matchup: int = 2,
    insurance_per_matchup: int = 1,
    min_pred_pts: float = 16.0,
    p_col: str | None = None,       # "p_over_22_5" OR None->use "p_over_baseline_3"
    min_prob: float | None = None,  # if None, auto-picks based on prob type
    min_delta_pts: float = 0.0,
    max_legs: int | None = None,
) -> pd.DataFrame:
    """
    Groups by matchup_key and takes top players_per_matchup + insurance_per_matchup.
    Ranks within matchup by: prob, delta_pts, pred_pts.
    """
    out = add_matchup_key(df)

    # choose prob column
    if p_col is None:
        if "p_over_baseline_3" in out.columns:
            p_col = "p_over_baseline_3"
        else:
            raise ValueError("p_col=None and 'p_over_baseline_3' not found. Pass p_col explicitly.")

    # choose default min_prob if not provided
    if min_prob is None:
        min_prob = 0.58 if p_col.startswith("p_over_") and "baseline" not in p_col else 0.18

    req = {"player","team","opp","is_home","pred_pts","delta_pts", p_col}
    _require_cols(out, req)

    pool = out[
        (out["pred_pts"] >= float(min_pred_pts)) &
        (out[p_col] >= float(min_prob)) &
        (out["delta_pts"] >= float(min_delta_pts))
    ].copy()

    if pool.empty:
        return pool

    pool = pool.sort_values(
        ["matchup_key", p_col, "delta_pts", "pred_pts"],
        ascending=[True, False, False, False],
    )
    pool["_rank"] = pool.groupby("matchup_key").cumcount()

    keep = int(players_per_matchup) + int(insurance_per_matchup)
    ticket = pool[pool["_rank"] < keep].copy()

    if max_legs is not None and len(ticket) > int(max_legs):
        ticket = ticket.sort_values([p_col, "delta_pts", "pred_pts"], ascending=False).head(int(max_legs))

    return (
        ticket
        .drop(columns=["_rank"])
        .sort_values(["matchup_key", p_col, "delta_pts", "pred_pts"], ascending=[True, False, False, False])
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# 4) Pencil labels (smash / over / coverage_only)
# ------------------------------------------------------------
def assign_pencil_decision_pts(
    df: pd.DataFrame,
    *,
    p_over_line_col: str | None = None,   # e.g. "p_over_22_5"
) -> pd.DataFrame:
    """
    Adds df['pencil'] based on a chosen probability column.
    Prefers sportsbook column if provided, else falls back to p_over_baseline_3.
    """
    out = df.copy()

    prob_col = None
    if p_over_line_col and p_over_line_col in out.columns:
        prob_col = p_over_line_col
    elif "p_over_baseline_3" in out.columns:
        prob_col = "p_over_baseline_3"

    if prob_col is None:
        out["pencil"] = "coverage_only"
        return out

    # tweak these however you want
    conditions = [
        (out[prob_col] >= 0.65) & (out["pred_pts"] >= 20.0),  # smash
        (out[prob_col] >= 0.58) & (out["pred_pts"] >= 16.0),  # over
    ]
    choices = ["smash", "over"]

    out["pencil"] = np.select(conditions, choices, default="coverage_only")
    return out


In [ ]:
_tdy = "2026-02-19"

# sportsbook prob file (recommended for "over line" style)
out_all = pd.read_csv(f"../results/{_tdy}/all_matchups_pts_p_over_22_5.csv")

ticket_over = select_over_line_ticket(out_all, n_legs=10, p_col="p_over_22_5", min_pred_pts=16, min_prob=0.58)
ticket_cov  = select_matchup_coverage_pts_ticket(out_all, players_per_matchup=2, insurance_per_matchup=1, p_col="p_over_22_5")

labeled = assign_pencil_decision_pts(out_all, p_over_line_col="p_over_22_5")

display(ticket_over.head(10))
display(ticket_cov.head(30))
display(labeled[["player","team","pred_pts","delta_pts","p_over_22_5","pencil"]].head(25))


,player,team,opp,is_home,pred_fg2a,pred_fg2_rate,pred_fta,pred_ft_rate,pred_fg3a,pred_fg3_rate,pred_pts,sd_pts,baseline_pts,delta_pts,p_over_baseline_3,p_over_22_5,matchup_key
0,Kawhi Leonard,LAC,DEN,1,11.980065,0.539431,6.219664,0.840508,6.640523,0.351944,25.163793,5.135214,27.024947,-1.861154,0.144251,0.731085,DEN@LAC
1,Tyrese Maxey,PHI,ATL,1,11.601117,0.503546,5.716504,0.762699,8.779905,0.347643,25.200188,5.527721,28.398735,-3.198546,0.107576,0.718683,ATL@PHI
2,Jalen Brunson,NYK,DET,1,11.449343,0.502393,6.449822,0.806183,7.088086,0.329909,23.719146,5.153589,26.680543,-2.961397,0.119762,0.630653,DET@NYK
3,Jaylen Brown,BOS,GSW,0,13.006610,0.513595,6.681735,0.644602,5.749826,0.349608,23.697867,5.127806,29.277930,-5.580063,0.040653,0.629719,BOS@GSW
4,Donovan Mitchell,CLE,BKN,1,9.693637,0.564590,5.950416,0.824583,7.962755,0.327473,23.675228,5.116213,28.190383,-4.515156,0.059070,0.628331,BKN@CLE


,player,team,opp,is_home,pred_fg2a,pred_fg2_rate,pred_fta,pred_ft_rate,pred_fg3a,pred_fg3_rate,pred_pts,sd_pts,baseline_pts,delta_pts,p_over_baseline_3,p_over_22_5,matchup_key


,player,team,pred_pts,delta_pts,p_over_22_5,pencil
0,Kevin Durant,HOU,21.930878,-3.067410,4.942523e-01,coverage_only
1,Alperen Şengün,HOU,16.687253,-4.438836,1.044466e-01,coverage_only
2,Amen Thompson,HOU,14.296351,-3.657819,1.883976e-02,coverage_only
3,Jabari Smith Jr.,HOU,12.967809,-2.302221,1.639665e-02,coverage_only
4,Reed Sheppard,HOU,11.857150,-0.844687,8.125609e-03,coverage_only
5,Tari Eason,HOU,11.467196,-0.547065,4.498715e-03,coverage_only
6,Donovan Mitchell,CLE,23.675228,-4.515156,6.283306e-01,over
7,James Harden,CLE,19.511586,-5.447884,3.044552e-01,coverage_only
8,Jaylon Tyson,CLE,11.113605,-1.926947,2.059004e-03,coverage_only
9,Jarrett Allen,CLE,11.054407,-2.862350,1.231176e-04,coverage_only


In [ ]:
temp = labeled[labeled["pencil"] != "coverage_only"]
display(temp[["player","team","pred_pts","delta_pts","p_over_22_5","pencil"]])


,player,team,pred_pts,delta_pts,p_over_22_5,pencil
6,Donovan Mitchell,CLE,23.675228,-4.515156,0.628331,over
13,Tyrese Maxey,PHI,25.200188,-3.198546,0.718683,smash
43,Jalen Brunson,NYK,23.719146,-2.961397,0.630653,over
80,Jaylen Brown,BOS,23.697867,-5.580063,0.629719,over
108,Kawhi Leonard,LAC,25.163793,-1.861154,0.731085,smash


In [ ]:
temp.sort_values(by="pred_pts", ascending=False)

,player,team,pred_pts,delta_pts,p_over_22_5,pencil
13,Tyrese Maxey,PHI,25.200188,-3.198546,7.186833e-01,smash
6,Donovan Mitchell,CLE,23.675228,-4.515156,6.283306e-01,over
0,Kevin Durant,HOU,21.930878,-3.067410,4.942523e-01,coverage_only
14,Jalen Johnson,ATL,21.126109,-1.898930,4.270824e-01,coverage_only
7,James Harden,CLE,19.511586,-5.447884,3.044552e-01,coverage_only
1,Alperen Şengün,HOU,16.687253,-4.438836,1.044466e-01,coverage_only
15,CJ McCollum,ATL,16.652729,-2.002811,1.289061e-01,coverage_only
16,Nickeil Alexander-Walker,ATL,16.487222,-3.436074,1.178263e-01,coverage_only
2,Amen Thompson,HOU,14.296351,-3.657819,1.883976e-02,coverage_only
17,Onyeka Okongwu,ATL,14.080804,-1.735359,3.126922e-02,coverage_only
